<a href="https://colab.research.google.com/github/lillymitch/organized_archives/blob/main/Final_Project_update.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

DATA_URL = "https://media.githubusercontent.com/media/metmuseum/openaccess/master/MetObjects.csv"

df = pd.read_csv(DATA_URL, low_memory=False)

# for demo
print("Original dataset size:", len(df))
print("Columns:", df.columns.tolist())
#

# remove missing medium
before = len(df)
df = df[df["Medium"].notna()]
after = len(df)

print("\nAfter removing missing medium:")
print(f"Removed: {before - after}")
print(f"Remaining: {after}")

## filter to common media
medium_counts = df["Medium"].value_counts()

print("\nTop 20 media BEFORE filtering:")
print(medium_counts.head(30))

valid_media = medium_counts[medium_counts >= 200].index

before = len(df)
df = df[df["Medium"].isin(valid_media)]
after = len(df)

print("\nAfter keeping common media (>=200 samples):")
print(f"Removed: {before - after}")
print(f"Remaining: {after}")
print(f"Number of media types before: {len(medium_counts)}")
print(f"Number of media types after: {len(valid_media)}")


Original dataset size: 484956
Columns: ['Object Number', 'Is Highlight', 'Is Timeline Work', 'Is Public Domain', 'Object ID', 'Gallery Number', 'Department', 'AccessionYear', 'Object Name', 'Title', 'Culture', 'Period', 'Dynasty', 'Reign', 'Portfolio', 'Constituent ID', 'Artist Role', 'Artist Prefix', 'Artist Display Name', 'Artist Display Bio', 'Artist Suffix', 'Artist Alpha Sort', 'Artist Nationality', 'Artist Begin Date', 'Artist End Date', 'Artist Gender', 'Artist ULAN URL', 'Artist Wikidata URL', 'Object Date', 'Object Begin Date', 'Object End Date', 'Medium', 'Dimensions', 'Credit Line', 'Geography Type', 'City', 'State', 'County', 'Country', 'Region', 'Subregion', 'Locale', 'Locus', 'Excavation', 'River', 'Classification', 'Rights and Reproduction', 'Link Resource', 'Object Wikidata URL', 'Metadata Date', 'Repository', 'Tags', 'Tags AAT URL', 'Tags Wikidata URL']

After removing missing medium:
Removed: 7215
Remaining: 477741

Top 20 media BEFORE filtering:
Medium
Terracotta    

In [2]:
# balance dataset
small_df = (
  df.groupby("Medium")
    .apply(lambda x: x.sample(min(len(x), 300), random_state=42))
    .reset_index(drop=True)
)

# for demo
print("\nAfter balancing:")
print("Dataset size:", len(small_df))
print("\nSamples per medium:")
print(small_df["Medium"].value_counts())


After balancing:
Dataset size: 61237

Samples per medium:
Medium
Albumen photograph                          300
Albumen silver print                        300
Albumen silver print from glass negative    300
Albumen silver prints                       300
Albumen silver print from paper negative    300
                                           ... 
Glazed earthenware                          203
Paper, silk                                 203
Mosaic glass                                203
Pen and india ink on paper                  202
Illustrations: color lithographs            200
Name: count, Length: 215, dtype: int64


/tmp/ipykernel_9074/2141898580.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 300), random_state=42))


In [3]:
def simplify_medium(m):
  if not isinstance(m, str):
    return None

  m = m.lower()

  if "terracotta" in m:
    return "terracotta"
  elif "lithograph" in m:
    return "lithograph"
  elif "etching" in m:
    return "etching"
  elif "engraving" in m:
    return "engraving"
  elif "gelatin silver print" in m:
    return "photograph"
  elif "albumen photograph" in m:
    return "photograph"
  elif "silk" in m:
    return "silk"
  elif "bronze" in m:
    return "bronze"
  elif "glass" in m:
    return "glass"
  else:
    return None

In [4]:
small_df["medium_simple"] = small_df["Medium"].apply(simplify_medium)

# for demo
print("\nExamples of medium simplification:")
print(small_df[["Medium", "medium_simple"]].head(10))

before = len(small_df)
small_df = small_df[small_df["medium_simple"].notna()]
after = len(small_df)

# for demo
print("\nAfter simplifying labels:")
print(f"Removed unclear media: {before - after}")
print(f"Remaining: {after}")



Examples of medium simplification:
  Medium medium_simple
0  Agate          None
1  Agate          None
2  Agate          None
3  Agate          None
4  Agate          None
5  Agate          None
6  Agate          None
7  Agate          None
8  Agate          None
9  Agate          None

After simplifying labels:
Removed unclear media: 42249
Remaining: 18988


In [5]:
import requests
# fetch image urls
def get_image_url(object_id):
  url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{int(object_id)}"

  try:
    r = requests.get(url, timeout=5)
    if r.status_code != 200:
      return None

    data = r.json()

    return data.get("primaryImageSmall") or data.get("primaryImage")

  except:
    return None

In [6]:
from concurrent.futures import ThreadPoolExecutor

def fetch_parallel(df, max_workers=10):
  with ThreadPoolExecutor(max_workers=max_workers) as executor:
    urls = list(executor.map(get_image_url, df["Object ID"]))

  df["image_url"] = urls
  return df

# for demo
print("\nFetching image URLs...")
small_df = fetch_parallel(small_df)



Fetching image URLs...


In [7]:
# label encoding
unique_media = small_df["medium_simple"].unique()
label_map = {medium: i for i, medium in enumerate(unique_media)}
small_df["label"] = small_df["medium_simple"].map(label_map)

print("\nLabel mapping:")
print(label_map)


Label mapping:
{'photograph': 0, 'glass': 1, 'bronze': 2, 'lithograph': 3, 'engraving': 4, 'etching': 5, 'silk': 6, 'terracotta': 7}


In [8]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import requests
from io import BytesIO
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

class MetDataset(Dataset):
  def __init__(self, df):
    self.df = df.reset_index(drop=True)

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]

    url = row["image_url"]
    label = row["label"]

    try:
      r = requests.get(url, timeout=5)
      img = Image.open(BytesIO(r.content)).convert("RGB")
      img = transform(img)
    except:
      img = torch.zeros((3, 224, 224))

    return img, torch.tensor(label)

dataset = MetDataset(small_df)

In [9]:
# Map 'medium_simple' categories to numerical labels and add a 'label' column
unique_media = small_df["medium_simple"].unique()
label_map = {medium: i for i, medium in enumerate(unique_media)}
small_df["label"] = small_df["medium_simple"].map(label_map)
num_classes = len(unique_media)

dataset = MetDataset(small_df)



In [10]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
  dataset,
  [train_size, val_size],
  generator=torch.Generator().manual_seed(42)
)

print(train_size, val_size)

15190 3798


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [12]:
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Define a simple Logistic Regression model for image data
class LogisticRegression(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Input image is 3x224x224. Flatten to 3 * 224 * 224 features.
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(3 * 224 * 224, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        return self.linear(x)

# Define the cross-entropy loss function
def cross_entropy_loss(outputs, targets):
    return F.cross_entropy(outputs, targets)

In [13]:
def test_optimizer(model_builder_function, n_epochs=10):
    # num_classes, cross_entropy_loss, and train_loader are
    # expected to be available globally or passed in.

    small_alpha = 1e-6
    large_alpha = 2e-5

    optimizers_config = {
        "Gradient descent": {'lr': small_alpha, 'momentum': 0.0, 'optimizer_class': torch.optim.SGD},
        "Gradient descent (high learning rate)": {'lr': large_alpha, 'momentum': 0.0, 'optimizer_class': torch.optim.SGD},
        "Momentum": {'lr': small_alpha, 'momentum': 0.9, 'optimizer_class': torch.optim.SGD},
        "Momentum (high learning rate)": {'lr': large_alpha, 'momentum': 0.9, 'optimizer_class': torch.optim.SGD},
        "Adam": {'lr': small_alpha, 'optimizer_class': torch.optim.Adam},
        "Adam (high learning rate)": {'lr': large_alpha, 'optimizer_class': torch.optim.Adam},
    }

    losses_dict = {}

    for name, config in optimizers_config.items():
        # Re-initialize the model for each optimizer test to ensure a clean slate
        model = model_builder_function() # Build the model using the provided function

        # Create optimizer instance, passing model.parameters()
        optimizer_params = {'lr': config['lr']}
        if 'momentum' in config:
            optimizer_params['momentum'] = config['momentum']
        optimizer = config['optimizer_class'](model.parameters(), **optimizer_params)

        losses_dict[name] = []
        for epoch in range(n_epochs):
            total_loss = 0
            # Iterate over the train_loader to get batches of images and labels
            for images, labels in train_loader:
                # Forward pass
                y_pred = model(images)
                # Calculate loss
                loss = cross_entropy_loss(y_pred, labels)

                # Backpropagation
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * images.size(0) # Aggregate loss weighted by batch size

            avg_epoch_loss = total_loss / len(train_loader.dataset) # Average loss for the epoch
            losses_dict[name].append(avg_epoch_loss)

    fig, axarr = plt.subplots(1, 3, figsize = (12, 4), sharey=True)
    for name, losses in losses_dict.items():
        if "Gradient descent" in name:
            i = 0
        elif "Momentum" in name:
            i = 1
        else: # Adam
            i = 2

        if "high learning rate" in name:
            label = name.split(" ")[0] + " (high lr)"
            alpha_val = large_alpha
        else:
            label = name.split(" ")[0]
            alpha_val = small_alpha

        axarr[i].plot(losses, label=r"$\alpha = $" + f"{alpha_val:.0e}", linestyle = '-' if "high learning rate" in name else '--')
        axarr[i].set_title(name.split(" ")[0])
        if i == 2: # Only show legend on the last subplot
            axarr[i].legend()

        if i == 0:
            axarr[i].set_ylabel("Loss")
    for ax in axarr:
        ax.set_xlabel("Epoch")
        ax.grid(True)

    plt.suptitle("Training Loss for Different Optimizers")
    plt.tight_layout()
    plt.show()

# Note: This call will now require a model_builder_function, which will be defined in the checklist.
# For example: test_optimizer(build_my_transfer_learning_model, n_epochs=10)